In [157]:
import numpy as np

In [158]:
board = np.full((3,3), " ")

In [159]:
def idx_to_row_colum(idx):
    row = idx // 3
    colum = idx % 3
    return row, colum

In [160]:
def play_to_board(board:np.ndarray, idx:int, value:str):
    x = board.copy()
    x[idx_to_row_colum(idx)] = value
    return x

In [161]:
def leagal_move_indicies(board:np.ndarray):
    return np.where((board.flatten()==" "))[0]

In [162]:
def is_game_over(board):
    lines = []

    # rows and columns
    lines.extend(board)
    lines.extend(board.T)

    # diagonals
    lines.append(np.diag(board))
    lines.append(np.diag(np.fliplr(board)))

    for line in lines:
        values = set(line)
        if values == {"X"}:
            return True, 1
        if values == {"O"}:
            return True, -1

    if len(leagal_move_indicies(board)) == 0:
        return True, 0

    return False, None

In [163]:
x_win_board = np.array([
    ["X", "O", "X"],
    ["O", "X", " "],
    ["X", " ", "O"],
])

unfinished_board = np.array([
    ["X", "O", "X"],
    ["O", "X", " "],
    [" ", " ", "O"],
])

about_to_win_board = np.array([
    ["X", " ", " "],
    ["O", " ", " "],
    ["X", "O", " "],
])

print(is_game_over(x_win_board))
print(is_game_over(unfinished_board))

(True, 1)
(False, None)


In [164]:
board = play_to_board(board, 4, "X")
print(leagal_move_indicies(board))

[0 1 2 3 5 6 7 8]


In [165]:
def minimax(depth, board, maximizing_player):
    res = is_game_over(board)
    if res[0]:
        return res[1]
    if depth == 0:
        return 0

    if maximizing_player:
        max_eval = -float("inf")
        moves = leagal_move_indicies(board)
        for move in moves:
            eval = minimax(depth-1, play_to_board(board, move, "X"), not maximizing_player)
            max_eval = max(eval, max_eval)
        return max_eval
    
    max_eval = float("inf")
    moves = leagal_move_indicies(board)
    for move in moves:
        eval = minimax(depth-1, play_to_board(board, move, "O"), not maximizing_player)
        max_eval = min(eval, max_eval)
    return max_eval

In [166]:
def board_telemetry(board):
    print("Fixed starting board:")
    print(board)
    print("Cell indices:")
    print(np.array([
        [0, 1, 2],
        [3, 4, 5],
        [6, 7, 8],
    ]))
    print("Legal X moves:", leagal_move_indicies(board))

In [167]:
minimax(30, about_to_win_board, True)

1

In [168]:
board_telemetry(about_to_win_board)

Fixed starting board:
[['X' ' ' ' ']
 ['O' ' ' ' ']
 ['X' 'O' ' ']]
Cell indices:
[[0 1 2]
 [3 4 5]
 [6 7 8]]
Legal X moves: [1 2 4 5 8]


In [169]:
test_board = np.array([
    ["X", "O", "X"],
    ["O", "X", " "],
    [" ", " ", "O"],
])

In [170]:
def telemetry_moves(board, turn="O", depth=30):
    move_scores = []
    for move in leagal_move_indicies(board):
        new_board = play_to_board(board, move, turn)
        next_player_is_x = turn == "O"
        score = minimax(depth=depth - 1, board=new_board, maximizing_player=next_player_is_x)
        move_scores.append({
            "move": int(move),
            "player": turn,
            "score_for_X": score,
        })

    return move_scores

In [171]:
print("Fixed starting board:")
print(test_board)

print("Cell indices:")
print(np.array([[0, 1, 2], [3, 4, 5], [6, 7, 8]]))
print("Legal X moves:", leagal_move_indicies(test_board))

scores = telemetry_moves(test_board, turn="X", depth=9)

print("X candidate move scores:")
for item in scores:
    print(f"move {item['move']} -> score_for_X {item['score_for_X']}")

best_for_x = max(scores, key=lambda item: item["score_for_X"])
print(f"Chosen X move: {best_for_x['move']} with score_for_X {best_for_x['score_for_X']}")

print("Explanation: X chooses the move with the highest minimax score because X is the maximizing player and assumes O responds optimally.")

Fixed starting board:
[['X' 'O' 'X']
 ['O' 'X' ' ']
 [' ' ' ' 'O']]
Cell indices:
[[0 1 2]
 [3 4 5]
 [6 7 8]]
Legal X moves: [5 6 7]
X candidate move scores:
move 5 -> score_for_X 0
move 6 -> score_for_X 1
move 7 -> score_for_X 0
Chosen X move: 6 with score_for_X 1
Explanation: X chooses the move with the highest minimax score because X is the maximizing player and assumes O responds optimally.


## CEOAI 1(b) Minimax Completion Check

This notebook covers CEOAI `1(b) Minimax and variations`.

Problem type: adversarial search. There are two players, `X` and `O`, and each player is assumed to choose the best move for their own outcome.

In this setup:
- `X` is the maximizing player.
- `O` is the minimizing player.
- terminal score `+1` means `X` wins.
- terminal score `0` means draw.
- terminal score `-1` means `O` wins.

The printed candidate move scores show the minimax value after each legal `O` move. `O` chooses the move with the lowest `score_for_X` because that is best for `O`.

Alpha-beta pruning is a faster version of minimax: it skips branches that cannot change the final chosen move, but it should return the same optimal move as plain minimax.